# W&B Sweep Sampling Playground

This notebook relies on the standalone `sweeps` library to reproduce wandb sweep sampling locally. This allows for the validation of parameter distributions before launching actual Hydra + W&B experiment sweeps.

**Setup:**
- Install notebook-specific dependencies if needed: `uv pip install "wandb[sweeps]"`.
- The example sweep exercises every active W&B sampler (constant, categorical, weighted categorical, uniform/int uniform, quantized uniform, log/uniform inverse variants, normal/log-normal, and beta families) so you can visually confirm each distribution.
- Adjust `SWEEP_CONFIG` to mirror your experiment preset as needed.


In [ ]:
import typing

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import sweeps
except ModuleNotFoundError as exc:  # guard for optional extra
    message = (
        "Install the sweeps extra with `pip install 'wandb[sweeps]'` or the "
        "standalone package via `pip install sweeps` before running this notebook."
    )
    raise ModuleNotFoundError(message) from exc

In [ ]:
SWEEP_CONFIG: dict[str, typing.Any] = {
    "name": "wandb-sweep-sampler-demo",
    "method": "random",
    "metric": {
        "name": "eval/loss",
        "goal": "minimize",
    },
    "parameters": {
        "constant_param": {
            "distribution": "constant",
            "value": 3,
        },
        "categorical_param": {
            "distribution": "categorical",
            "values": ["red", "green", "blue"],
        },
        "categorical_weighted_param": {
            "distribution": "categorical_w_probabilities",
            "values": ["cat", "dog", "bird"],
            "probabilities": [0.6, 0.3, 0.1],
        },
        "int_uniform_param": {
            "distribution": "int_uniform",
            "min": 1,
            "max": 6,
        },
        "uniform_param": {
            "distribution": "uniform",
            "min": -1.0,
            "max": 1.0,
        },
        "q_uniform_param": {
            "distribution": "q_uniform",
            "min": 0.0,
            "max": 12.0,
            "q": 2.0,
        },
        "log_uniform_values_param": {
            "distribution": "log_uniform_values",
            "min": 0.0005,
            "max": 1.0,
        },
        "q_log_uniform_values_param": {
            "distribution": "q_log_uniform_values",
            "min": 0.001,
            "max": 0.5,
            "q": 0.05,
        },
        "inv_log_uniform_values_param": {
            "distribution": "inv_log_uniform_values",
            "min": 0.2,
            "max": 5.0,
        },
        "normal_param": {
            "distribution": "normal",
            "mu": 0.0,
            "sigma": 1.0,
        },
        "q_normal_param": {
            "distribution": "q_normal",
            "mu": 3.0,
            "sigma": 1.5,
            "q": 0.25,
        },
        "log_normal_param": {
            "distribution": "log_normal",
            "mu": -0.2,
            "sigma": 0.5,
        },
        "q_log_normal_param": {
            "distribution": "q_log_normal",
            "mu": -0.2,
            "sigma": 0.5,
            "q": 0.1,
        },
        "beta_param": {
            "distribution": "beta",
            "a": 2.0,
            "b": 5.0,
        },
        "q_beta_param": {
            "distribution": "q_beta",
            "a": 2.0,
            "b": 5.0,
            "q": 0.05,
        },
    },
}


def sample_sweep_runs(
    sweep_config: dict[str, typing.Any],
    total_runs: int,
) -> pd.DataFrame:
    """Draws sweep parameter assignments using the sweeps sampler."""
    sweep_definition = sweeps.SweepConfig(sweep_config)
    sweep_runs = sweeps.next_runs(sweep_definition, [], n=total_runs)
    records: list[dict[str, typing.Any]] = []

    def extract_value(sample: typing.Any) -> typing.Any:
        if isinstance(sample, dict) and "value" in sample and len(sample) == 1:
            sample = sample["value"]
        if hasattr(sample, "item") and callable(sample.item):
            try:
                return sample.item()
            except Exception:
                return sample
        return sample

    for run_idx, run in enumerate(sweep_runs):
        record = {"sample_index": run_idx}
        for key, value in run.config.items():
            record[key] = extract_value(value)
        records.append(record)
    return pd.DataFrame.from_records(records)


def plot_parameter_distributions(
    dataframe: pd.DataFrame,
    sweep_config: dict[str, typing.Any],
) -> None:
    """Visualizes sampled parameter distributions for quick inspection."""
    parameter_names = list(sweep_config.get("parameters", {}).keys())
    if not parameter_names:
        raise ValueError("sweep_config must define at least one parameter")

    figure, axes = plt.subplots(len(parameter_names), 1, figsize=(8.5, 3.6 * len(parameter_names)))
    axes_array = np.atleast_1d(axes)

    for axis, parameter_name in zip(axes_array, parameter_names, strict=False):
        series = dataframe[parameter_name]
        parameter_definition = sweep_config["parameters"].get(parameter_name, {})
        distribution_kind = str(parameter_definition.get("distribution", "")).lower()

        try:
            numeric_series = pd.to_numeric(series)
        except (TypeError, ValueError):
            numeric_series = series

        if pd.api.types.is_numeric_dtype(numeric_series):
            values = numeric_series.to_numpy()
            unique_count = len(pd.unique(values))
            bin_count = min(60, max(5, int(np.sqrt(values.size)))) if unique_count > 3 else unique_count
            axis.hist(values, bins=bin_count, color="#2C7BB6", edgecolor="#FFFFFF", alpha=0.85)
            if "log" in distribution_kind and (values > 0).all():
                axis.set_xscale("log")
        else:
            counts = numeric_series.astype(str).value_counts(sort=False)
            axis.bar(counts.index.tolist(), counts.values.tolist(), color="#2C7BB6", alpha=0.85)
            axis.set_ylim(0, counts.values.max() * 1.15)
            axis.tick_params(axis="x", labelrotation=45)
        axis.set_title(parameter_name)
        axis.set_ylabel("count")
        axis.grid(True, axis="y", alpha=0.25)

    axes_array[-1].set_xlabel("sampled values")
    for ax in axes_array:
        ax.margins(x=0.05)
        ax.tick_params(axis="x", labelrotation=45)
    figure.tight_layout()

In [ ]:
SAMPLE_COUNT = 10_000
SAMPLES = sample_sweep_runs(SWEEP_CONFIG, SAMPLE_COUNT)
SAMPLES.describe(include="all")

In [ ]:
plot_parameter_distributions(SAMPLES, SWEEP_CONFIG)